# Coding Homework 03: Vector Spaces, Inner Products, and Attention

In this assignment you will turn the geometry of vector spaces into the
central computation performed by a transformer attention head.

By the end, you will be able to:

- test whether column vectors are linearly independent;
- compute a projection and cosine similarity;
- interpret every entry of $QK^\top$ as an inner product;
- turn scores into stable row-wise softmax weights;
- use a causal mask so a token cannot read future tokens; and
- add a reusable attention head to your cumulative `mini_transformer.py`
  model.

**No prior data-science or AI background is assumed.** New vocabulary is
defined before it is used.

**Estimated time:** 2.5–3.5 hours.


## Submission and grading

Keep these two files in the same folder while you work:

1. `CodingHW_03_VectorSpaces_InnerProducts.ipynb`
2. `mini_transformer.py`

Submit **both files** to the course portal. Run the entire notebook from a fresh
kernel before submitting. The tokenizer helpers are supplied and ungraded. The autograder uses additional numerical cases,
so matching one displayed example is not enough.

| Component | Points |
|---|---:|
| Linear independence | 1 |
| Projection and cosine similarity | 4 |
| Scaled dot-product scores | 3 |
| Stable softmax | 3 |
| Unmasked and causally masked attention | 5 |
| Reusable attention head in `mini_transformer.py` | 4 |
| **Total** | **20** |

**Connection to written HW03.** Written Problem 2 develops span,
independence, and coordinates; Problem 5 interprets token embeddings and
cosine similarity; Problem 6 computes attention by hand. This notebook
performs the same mathematics numerically.


## Setup


In [1]:
import importlib
import numpy as np
import mini_transformer as model

def reload_model():
    # Reload mini_transformer.py after you save changes to it.
    global model
    importlib.invalidate_caches()
    model = importlib.reload(model)
    return model

np.set_printoptions(precision=4, suppress=True)
print("Setup complete — using", model.__file__)


Setup complete — using /agent/repos/Applied-AI-ML-projects/coursework/MATH412/notebooks/mini_transformer.py


---
## Part 1: A supplied bridge from text to token IDs

A **token** is a unit of text represented by an integer ID. Real language
models use sophisticated tokenizers; here we deliberately use lowercase
words separated by spaces so that the linear algebra stays visible.

The two helper functions below are **provided**. You are not being asked to
write or debug tokenization code. Run them and use the output to see how text
becomes numerical input for a model:

- `tokenize` lowercases and splits a sentence, then looks up each word;
- an unfamiliar word receives the `[UNK]` (“unknown”) ID; and
- `detokenize` converts the IDs back to readable tokens.

The mathematical work begins in Part 2.


In [2]:
def tokenize(sentence, vocab):
    """Convert a space-separated sentence to token IDs (supplied)."""
    words = sentence.lower().split()
    unknown_id = vocab["[UNK]"]
    return [vocab.get(word, unknown_id) for word in words]

def detokenize(token_ids, inverse_vocab):
    """Convert token IDs back to a space-separated string (supplied)."""
    return " ".join(inverse_vocab[token_id] for token_id in token_ids)


In [3]:
vocab = {"[UNK]": 0, "the": 1, "cat": 2, "sat": 3, "on": 4, "mat": 5}
inverse_vocab = {token_id: token for token, token_id in vocab.items()}

ids = tokenize("The cat danced", vocab)
print("Token IDs:", ids)
print("Decoded:  ", detokenize(ids, inverse_vocab))
assert ids == [1, 2, 0]
assert detokenize(ids, inverse_vocab) == "the cat [UNK]"


Token IDs: [1, 2, 0]
Decoded:   the cat [UNK]


---
## Part 2: Vector-space geometry in NumPy

### Rows versus columns

In a linear-algebra expression such as $Bc=x$, the vectors being combined
are stored as the **columns** of $B$. In a data matrix or transformer, one
observation or token is usually stored in each **row**. Neither convention
is intrinsically correct; always identify what an axis represents.

The rank of a matrix is the number of independent columns. Therefore the
columns of an $m\times n$ matrix are linearly independent exactly when its
rank is $n$.


In [4]:
def is_linearly_independent(B):
    """Return True exactly when the columns of B are independent."""
    B = np.asarray(B, dtype=float)
    number_of_columns = B.shape[1]
    rank = np.linalg.matrix_rank(B)

    return rank == number_of_columns


In [5]:
B = np.array([[1., 1., 0.],
              [1., 0., 1.],
              [0., 1., 1.]])
dependent_B = np.column_stack((B[:, 0], B[:, 1], B[:, 0] + B[:, 1]))

assert is_linearly_independent(B)
assert not is_linearly_independent(dependent_B)
print("rank(B) =", np.linalg.matrix_rank(B))
print("rank(dependent_B) =", np.linalg.matrix_rank(dependent_B))


rank(B) = 3
rank(dependent_B) = 2


### Projection and cosine similarity

For a nonzero direction $u$, the projection of $x$ onto
$\operatorname{span}\{u\}$ is

$$\operatorname{proj}_u(x)=\frac{x^\top u}{u^\top u}u.$$

The residual $x-\operatorname{proj}_u(x)$ must be orthogonal to $u$.
Cosine similarity normalizes an inner product by both lengths:

$$\cos(x,y)=\frac{x^\top y}{\|x\|\,\|y\|}.$$

It compares direction rather than magnitude. Neither formula is defined
when its required vector has norm zero; raise `ValueError` in that case.


In [6]:
def project_onto_vector(x, u):
    """Return the projection of x onto span{u}."""
    x = np.asarray(x, dtype=float)
    u = np.asarray(u, dtype=float)
    denominator = u @ u

    # This check is supplied because projection onto the zero vector is undefined.
    if np.isclose(denominator, 0.0):
        raise ValueError("Cannot project onto the zero vector")

    coefficient = (x @ u) / denominator

    return coefficient * u

def cosine_similarity(x, y):
    """Return the cosine of the angle between nonzero vectors x and y."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_norm = np.linalg.norm(x)
    y_norm = np.linalg.norm(y)

    # This check is supplied because the formula would divide by zero.
    if np.isclose(x_norm, 0.0) or np.isclose(y_norm, 0.0):
        raise ValueError("Cosine similarity requires nonzero vectors")

    return (x @ y) / (x_norm * y_norm)


In [7]:
x = np.array([3., 1., 2.])
u = np.array([1., 1., 0.])
projection = project_onto_vector(x, u)
residual = x - projection

print("projection:", projection)
print("residual:  ", residual)
assert np.allclose(projection, [2., 2., 0.])
assert np.isclose(residual @ u, 0.0)
assert np.isclose(cosine_similarity(u, 5*u), 1.0)
assert np.isclose(cosine_similarity([1, 0], [0, 1]), 0.0)


projection: [2. 2. 0.]
residual:   [ 1. -1.  2.]


---
## Part 3: Inner products become attention

A transformer assigns each token three row vectors:

- a **query**: what information this token is looking for;
- a **key**: what information this token offers; and
- a **value**: the information to be combined.

If `Q` has $n_q$ rows and `K` has $n_k$ rows, then

$$S=\frac{QK^\top}{\sqrt{d_k}}\in\mathbb R^{n_q\times n_k}.$$

Entry $S_{ij}$ is the scaled inner product between query $i$ and key $j$.
The query and key counts need not be equal, but their feature dimension
$d_k$ must match.

Open `mini_transformer.py` and complete `scaled_dot_product_scores`.
Keep that file as the single source of your transformer implementation;
the notebook will call the function from the module.


In [8]:
model = reload_model()

Q_demo = np.array([[2., 0.],
                   [0., 2.],
                   [2., 2.]])
K_demo = Q_demo.copy()
scores_demo = model.scaled_dot_product_scores(Q_demo, K_demo)

print("Scores:\n", scores_demo)
assert scores_demo.shape == (3, 3)
assert np.allclose(scores_demo, Q_demo @ K_demo.T / np.sqrt(2))


Scores:
 [[2.8284 0.     2.8284]
 [0.     2.8284 2.8284]
 [2.8284 2.8284 5.6569]]


Scores are not yet weights: they may be negative and do not sum to one.
Softmax converts the last axis of an array into positive weights:

$$\operatorname{softmax}(s)_j=\frac{e^{s_j}}{\sum_k e^{s_k}}.$$

Subtracting the largest entry before exponentiating changes neither the
result nor the ratios of the weights, but prevents numerical overflow.
Using `axis=-1` makes the function work on one vector and on every row of
a matrix.

Complete `softmax` in `mini_transformer.py`, save the file, and run the
next cell.


In [9]:
model = reload_model()
weights_demo = model.softmax(scores_demo)
vector_weights = model.softmax(np.array([1000., 1002., 1001.]))

print("Weights:\n", weights_demo)
print("Row sums:", weights_demo.sum(axis=1))
assert np.allclose(weights_demo.sum(axis=1), 1.0)
assert vector_weights.shape == (3,)
assert np.all(np.isfinite(vector_weights))
assert np.isclose(vector_weights.sum(), 1.0)
# Unlike tiny random initializations, this example visibly favors some keys.
assert weights_demo.max() > 0.75


Weights:
 [[0.4856 0.0287 0.4856]
 [0.0287 0.4856 0.4856]
 [0.0529 0.0529 0.8943]]
Row sums: [1. 1. 1.]


### Weighted values and causal masking

Attention outputs the weighted average

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}(S)V.$$

In a next-token model, token $i$ must not read a future token $j>i$.
Recall your HW01 `causal_mask(n)`: it is one on and below the diagonal and
zero above it. A mask is applied **before softmax** by replacing forbidden
scores with $-\infty$. Softmax then gives those positions weight zero.

`mask=None` means ordinary unmasked attention. Otherwise, require the mask
to have the same shape as the score matrix and require at least one allowed
key in every row.

Complete `attention` in `mini_transformer.py`. The supplied
`model.causal_mask` is the cumulative component from Coding HW01.


In [10]:
model = reload_model()
V_demo = np.array([[1., 0.],
                   [0., 1.],
                   [1., 1.]])
full_output = model.attention(Q_demo, K_demo, V_demo)
causal = model.causal_mask(3)
causal_output = model.attention(Q_demo, K_demo, V_demo, mask=causal)

print("Causal mask:\n", causal)
print("Unmasked output:\n", full_output)
print("Causal output:\n", causal_output)
assert np.allclose(causal_output[0], V_demo[0])
assert not np.allclose(full_output[0], causal_output[0])


Causal mask:
 [[1 0 0]
 [1 1 0]
 [1 1 1]]
Unmasked output:
 [[0.9713 0.5144]
 [0.5144 0.9713]
 [0.9471 0.9471]]
Causal output:
 [[1.     0.    ]
 [0.0558 0.9442]
 [0.9471 0.9471]]


**Explain in 2–3 sentences:** Why must the mask be applied before softmax?
What information would leak if the first token could use the third token's
value while predicting the next token?


**Your answer:**

_ The mask is be applied before because softmax normalizes values. If masked after, the normalization would distribute the probabilities incorrectly, among the allowed tokens. It would leak future information, making the model unrealistic for generating text._


---
## Part 4: Mini-transformer milestone — a reusable attention head

The query, key, and value matrices are not stored directly. They are
produced from a token matrix $X\in\mathbb R^{n\times d_{model}}$ by learned
linear maps:

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.$$

A single attention head is therefore

$$X\longmapsto\operatorname{Attention}(XW_Q,XW_K,XW_V).$$

You have already built the three calculations needed by the head. Complete
`attention_head` in `mini_transformer.py` by forming $Q$, $K$, and $V$ and
then calling your `attention` function. This module—not the notebook—is the
cumulative model that later assignments will extend.


In [11]:
model = reload_model()

X = np.array([[2., 0., 0., 0.],
              [0., 2., 0., 0.],
              [2., 2., 0., 0.]])
W_Q = np.array([[1., 0.], [0., 1.], [0., 0.], [0., 0.]])
W_K = W_Q.copy()
W_V = np.array([[0.5, 0.0], [0.0, 0.5], [0.0, 0.0], [0.0, 0.0]])

expected = model.attention(
    X @ W_Q, X @ W_K, X @ W_V, mask=model.causal_mask(3)
)
actual = model.attention_head(
    X, W_Q, W_K, W_V, mask=model.causal_mask(3)
)

print("Attention-head output:\n", actual)
assert actual.shape == (3, 2)
assert np.allclose(actual, expected)


Attention-head output:
 [[1.     0.    ]
 [0.0558 0.9442]
 [0.9471 0.9471]]


### What you have built

The milestone now has a continuous mathematical story:

1. HW01 represented tokens by rows and built the causal mask.
2. Units 8–10 supplied span, independence, projections, and inner products.
3. This assignment turned query–key inner products into weights and weighted
   averages of value vectors.
4. A later multi-head layer will run several such heads in parallel and
   concatenate their outputs.

Before submitting, restart the kernel and run all cells. Confirm that:

- [ ] all displayed assertions pass;
- [ ] `softmax` works for both 1-D and 2-D inputs;
- [ ] masked positions receive exactly zero attention weight;
- [ ] `model.attention_head` agrees with the component calculation; and
- [ ] both required files are uploaded.
